In [57]:
import cv2
import mediapipe as mp
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import Adam
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import os
import glob
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# ============================================================================
# CONFIG
# ============================================================================
VIDEO_DIR = r"C:\Users\PRASHANTH\gcn"
SEQ_LEN = 30
BATCH_SIZE = 16
EPOCHS = 100
LEARNING_RATE = 3e-4

# Graph connections for 17-landmark skeleton
ADJ = [
    (0,1),(0,2),(1,3),(2,4), (5,6),(5,7),(7,9),(6,8),(8,10),
    (11,12),(11,13),(13,15),(12,14),(14,16), (5,11),(6,12)
]

def get_adj_matrix():
    A = np.eye(17)
    for i, j in ADJ:
        A[i, j] = A[j, i] = 1
    D = np.diag(np.sum(A, axis=1)**-0.5)
    A_norm = D @ A @ D
    return torch.FloatTensor(A_norm)

# ============================================================================
# AUGMENTATION & PREPROCESSING
# ============================================================================
class SkeletonUtils:
    @staticmethod
    def normalize(coords):
        for i in range(len(coords)):
            mid_hip = (coords[i, 11, :] + coords[i, 12, :]) / 2
            coords[i] -= mid_hip
        return coords

    @staticmethod
    def add_noise(x, sigma=0.02):
        noise = torch.randn_like(x) * sigma
        return x + noise

    @staticmethod
    def augment(x):
        x = SkeletonUtils.add_noise(x)
        if torch.rand(1) > 0.5:
            x[:, :, [0]] = -x[:, :, [0]]
        scale = 1 + (torch.rand(1) - 0.5) * 0.2
        x = x * scale
        return x

    @staticmethod
    def calculate_angle(a, b, c):
        a, b, c = np.array(a), np.array(b), np.array(c)
        ba = a - b
        bc = c - b
        cos_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
        angle = np.arccos(np.clip(cos_angle, -1.0, 1.0))
        return np.degrees(angle)

# ============================================================================
# SQUAT PROCESSOR
# ============================================================================
class SquatProcessor:
    def __init__(self):
        self.pose = mp.solutions.pose.Pose(static_image_mode=False, min_detection_confidence=0.5)

    def get_si(self, left_angle, right_angle):
        return ((left_angle - right_angle) / (0.5 * (left_angle + right_angle) + 1e-6)) * 100

    def process_video(self, path):
        cap = cv2.VideoCapture(path)
        coords = []
        frame_labels = []
        video_id = os.path.basename(path)

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            res = self.pose.process(img)

            if res.pose_landmarks and res.pose_world_landmarks:
                pts = [[lm.x, lm.y, lm.z] for lm in res.pose_landmarks.landmark[12:29]]
                coords.append(pts)

                wlm = res.pose_world_landmarks.landmark
                left_knee = SkeletonUtils.calculate_angle(
                    [wlm[23].x, wlm[23].y, wlm[23].z],
                    [wlm[25].x, wlm[25].y, wlm[25].z],
                    [wlm[27].x, wlm[27].y, wlm[27].z]
                )
                right_knee = SkeletonUtils.calculate_angle(
                    [wlm[24].x, wlm[24].y, wlm[24].z],
                    [wlm[26].x, wlm[26].y, wlm[26].z],
                    [wlm[28].x, wlm[28].y, wlm[28].z]
                )
                si = self.get_si(left_knee, right_knee)
                frame_labels.append(1 if si > 10 else 0)

        cap.release()
        if len(coords) < SEQ_LEN:
            return None, None, None

        coords = SkeletonUtils.normalize(np.array(coords))
        final_label = 1 if np.mean(frame_labels) > 0.4 else 0
        return coords, final_label, video_id

# ============================================================================
# CTR-GCN MODEL
# ============================================================================
class CTR_GC(nn.Module):
    def __init__(self, in_c, out_c, adj):
        super().__init__()
        self.adj = nn.Parameter(adj, requires_grad=False)
        self.refine = nn.Conv2d(in_c, out_c, 1)
        self.pa = nn.Parameter(torch.randn(out_c, 17, 17) * 0.01)
        self.alpha = nn.Parameter(torch.ones(1))

    def forward(self, x):
        x = self.refine(x)
        A = self.alpha * self.adj.unsqueeze(0) + self.pa
        x = torch.einsum('bctv,cvw->bctw', x, A)
        return x

class CTRGCN_Block(nn.Module):
    def __init__(self, in_c, out_c, adj, stride=1):
        super().__init__()
        self.gcn = CTR_GC(in_c, out_c, adj)
        self.tcn = nn.Sequential(
            nn.BatchNorm2d(out_c),
            nn.ReLU(),
            nn.Conv2d(out_c, out_c, (9, 1), (stride, 1), (4, 0)),
            nn.BatchNorm2d(out_c),
            nn.Dropout(0.6)
        )
        self.res = nn.Conv2d(in_c, out_c, 1) if in_c != out_c else nn.Identity()
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(self.tcn(self.gcn(x)) + self.res(x))

class CTRGCN(nn.Module):
    def __init__(self):
        super().__init__()
        adj = get_adj_matrix()
        self.layer1 = CTRGCN_Block(3, 32, adj)
        self.dropout = nn.Dropout(0.5)
        self.layer2 = CTRGCN_Block(32, 16, adj)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Dropout(0.6),
            nn.Linear(16, 2)
        )

    def forward(self, x):
        x = self.layer1(x)
        x = self.dropout(x)
        x = self.layer2(x)
        x = self.pool(x).view(x.size(0), -1)
        return self.fc(x)

# ============================================================================
# EVALUATION
# ============================================================================
def evaluate(model, loader, device, criterion=None):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            if criterion:
                loss = criterion(output, target)
                total_loss += loss.item()
            all_preds.extend(torch.argmax(output, 1).cpu().numpy())
            all_labels.extend(target.cpu().numpy())
    return accuracy_score(all_labels, all_preds), total_loss/len(loader) if criterion else 0, all_labels, all_preds

# ============================================================================
# 10-FOLD CROSS VALIDATION (No Early Stopping)
# ============================================================================
def cross_validate_no_early_stop(X, y, folds=10):
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=42)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    fold_metrics = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        print(f"\n--- Fold {fold+1}/{folds} ---")

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        train_loader = DataLoader(list(zip(X_train, y_train)), BATCH_SIZE, shuffle=True)
        test_loader = DataLoader(list(zip(X_test, y_test)), BATCH_SIZE)

        model = CTRGCN().to(device)
        criterion = nn.CrossEntropyLoss(label_smoothing=0.20)
        optimizer = Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=5e-2)

        # TRAINING LOOP
        for epoch in range(EPOCHS):
            model.train()
            for data, target in train_loader:
                data = SkeletonUtils.augment(data)
                data, target = data.to(device), target.to(device)
                optimizer.zero_grad()
                output = model(data)
                loss = criterion(output, target)
                loss.backward()
                optimizer.step()

            if (epoch+1) % 10 == 0 or epoch == 0:
                train_acc, _, _, _ = evaluate(model, train_loader, device)
                print(f"Epoch {epoch+1}/{EPOCHS} | Train Acc: {train_acc:.2%}")

        # Evaluate on test set
        test_acc, _, y_true, y_pred = evaluate(model, test_loader, device)
        print(f"Fold {fold+1} Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
        fold_metrics.append(test_acc)

    print(f"\nAverage Test Accuracy over {folds} folds: {np.mean(fold_metrics):.4f} ({np.mean(fold_metrics)*100:.2f}%)")
    print(f"Standard Deviation: {np.std(fold_metrics):.4f}")

# ============================================================================
# MAIN FUNCTION
# ============================================================================
def main_cv_no_early_stop():
    proc = SquatProcessor()
    videos = glob.glob(os.path.join(VIDEO_DIR, "*.mp4"))
    X, y = [], []

    print("Extracting features from videos...")
    for v in videos:
        coords, label, _ = proc.process_video(v)
        if coords is not None:
            for i in range(0, len(coords) - SEQ_LEN, 10):
                X.append(coords[i:i+SEQ_LEN].transpose(2, 0, 1))
                y.append(label)

    if not X:
        print("No valid data found. Check your VIDEO_DIR path.")
        return

    X = torch.FloatTensor(np.array(X))
    y = torch.LongTensor(np.array(y))

    cross_validate_no_early_stop(X, y, folds=10)

# ============================================================================
# RUN
# ============================================================================
if __name__ == "__main__":
    main_cv_no_early_stop()

Extracting features from videos...

--- Fold 1/10 ---
Epoch 1/100 | Train Acc: 27.48%
Epoch 10/100 | Train Acc: 67.94%
Epoch 20/100 | Train Acc: 70.99%
Epoch 30/100 | Train Acc: 94.66%
Epoch 40/100 | Train Acc: 96.18%
Epoch 50/100 | Train Acc: 76.34%
Epoch 60/100 | Train Acc: 99.24%
Epoch 70/100 | Train Acc: 99.24%
Epoch 80/100 | Train Acc: 100.00%
Epoch 90/100 | Train Acc: 97.71%
Epoch 100/100 | Train Acc: 100.00%
Fold 1 Test Accuracy: 1.0000 (100.00%)

--- Fold 2/10 ---
Epoch 1/100 | Train Acc: 28.24%
Epoch 10/100 | Train Acc: 66.41%
Epoch 20/100 | Train Acc: 75.57%
Epoch 30/100 | Train Acc: 93.89%
Epoch 40/100 | Train Acc: 93.89%
Epoch 50/100 | Train Acc: 80.92%
Epoch 60/100 | Train Acc: 96.95%
Epoch 70/100 | Train Acc: 97.71%
Epoch 80/100 | Train Acc: 99.24%
Epoch 90/100 | Train Acc: 94.66%
Epoch 100/100 | Train Acc: 97.71%
Fold 2 Test Accuracy: 0.8667 (86.67%)

--- Fold 3/10 ---
Epoch 1/100 | Train Acc: 71.76%
Epoch 10/100 | Train Acc: 69.47%
Epoch 20/100 | Train Acc: 85.50%
Epoch

In [58]:
import cv2
import mediapipe as mp
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import Adam
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import os
import glob
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# ============================================================================
# CONFIG
# ============================================================================
VIDEO_DIR = r"C:\Users\PRASHANTH\gcn"
SEQ_LEN = 30
BATCH_SIZE = 16
EPOCHS = 100
LEARNING_RATE = 3e-4

# Graph connections for 17-landmark skeleton
ADJ = [
    (0,1),(0,2),(1,3),(2,4), (5,6),(5,7),(7,9),(6,8),(8,10),
    (11,12),(11,13),(13,15),(12,14),(14,16), (5,11),(6,12)
]

def get_adj_matrix():
    A = np.eye(17)
    for i, j in ADJ:
        A[i, j] = A[j, i] = 1
    D = np.diag(np.sum(A, axis=1)**-0.5)
    A_norm = D @ A @ D
    return torch.FloatTensor(A_norm)

# ============================================================================
# AUGMENTATION & PREPROCESSING
# ============================================================================
class SkeletonUtils:
    @staticmethod
    def normalize(coords):
        for i in range(len(coords)):
            mid_hip = (coords[i, 11, :] + coords[i, 12, :]) / 2
            coords[i] -= mid_hip
        return coords

    @staticmethod
    def add_noise(x, sigma=0.02):
        noise = torch.randn_like(x) * sigma
        return x + noise

    @staticmethod
    def augment(x):
        x = SkeletonUtils.add_noise(x)
        if torch.rand(1) > 0.5:
            x[:, :, [0]] = -x[:, :, [0]]
        scale = 1 + (torch.rand(1) - 0.5) * 0.2
        x = x * scale
        return x

    @staticmethod
    def calculate_angle(a, b, c):
        a, b, c = np.array(a), np.array(b), np.array(c)
        ba = a - b
        bc = c - b
        cos_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
        angle = np.arccos(np.clip(cos_angle, -1.0, 1.0))
        return np.degrees(angle)

# ============================================================================
# SQUAT PROCESSOR
# ============================================================================
class SquatProcessor:
    def __init__(self):
        self.pose = mp.solutions.pose.Pose(static_image_mode=False, min_detection_confidence=0.5)

    def get_si(self, left_angle, right_angle):
        return ((left_angle - right_angle) / (0.5 * (left_angle + right_angle) + 1e-6)) * 100

    def process_video(self, path):
        cap = cv2.VideoCapture(path)
        coords = []
        frame_labels = []
        video_id = os.path.basename(path)

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            res = self.pose.process(img)

            if res.pose_landmarks and res.pose_world_landmarks:
                pts = [[lm.x, lm.y, lm.z] for lm in res.pose_landmarks.landmark[12:29]]
                coords.append(pts)

                wlm = res.pose_world_landmarks.landmark
                left_knee = SkeletonUtils.calculate_angle(
                    [wlm[23].x, wlm[23].y, wlm[23].z],
                    [wlm[25].x, wlm[25].y, wlm[25].z],
                    [wlm[27].x, wlm[27].y, wlm[27].z]
                )
                right_knee = SkeletonUtils.calculate_angle(
                    [wlm[24].x, wlm[24].y, wlm[24].z],
                    [wlm[26].x, wlm[26].y, wlm[26].z],
                    [wlm[28].x, wlm[28].y, wlm[28].z]
                )
                si = self.get_si(left_knee, right_knee)
                frame_labels.append(1 if si > 10 else 0)

        cap.release()
        if len(coords) < SEQ_LEN:
            return None, None, None

        coords = SkeletonUtils.normalize(np.array(coords))
        final_label = 1 if np.mean(frame_labels) > 0.4 else 0
        return coords, final_label, video_id

# ============================================================================
# CTR-GCN MODEL
# ============================================================================
class CTR_GC(nn.Module):
    def __init__(self, in_c, out_c, adj):
        super().__init__()
        self.adj = nn.Parameter(adj, requires_grad=False)
        self.refine = nn.Conv2d(in_c, out_c, 1)
        self.pa = nn.Parameter(torch.randn(out_c, 17, 17) * 0.01)
        self.alpha = nn.Parameter(torch.ones(1))

    def forward(self, x):
        x = self.refine(x)
        A = self.alpha * self.adj.unsqueeze(0) + self.pa
        x = torch.einsum('bctv,cvw->bctw', x, A)
        return x

class CTRGCN_Block(nn.Module):
    def __init__(self, in_c, out_c, adj, stride=1):
        super().__init__()
        self.gcn = CTR_GC(in_c, out_c, adj)
        self.tcn = nn.Sequential(
            nn.BatchNorm2d(out_c),
            nn.ReLU(),
            nn.Conv2d(out_c, out_c, (9, 1), (stride, 1), (4, 0)),
            nn.BatchNorm2d(out_c),
            nn.Dropout(0.6)
        )
        self.res = nn.Conv2d(in_c, out_c, 1) if in_c != out_c else nn.Identity()
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(self.tcn(self.gcn(x)) + self.res(x))

class CTRGCN(nn.Module):
    def __init__(self):
        super().__init__()
        adj = get_adj_matrix()
        self.layer1 = CTRGCN_Block(3, 32, adj)
        self.dropout = nn.Dropout(0.5)
        self.layer2 = CTRGCN_Block(32, 16, adj)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Dropout(0.6),
            nn.Linear(16, 2)
        )

    def forward(self, x):
        x = self.layer1(x)
        x = self.dropout(x)
        x = self.layer2(x)
        x = self.pool(x).view(x.size(0), -1)
        return self.fc(x)

# ============================================================================
# EVALUATION
# ============================================================================
def evaluate(model, loader, device, criterion=None):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            if criterion:
                loss = criterion(output, target)
                total_loss += loss.item()
            all_preds.extend(torch.argmax(output, 1).cpu().numpy())
            all_labels.extend(target.cpu().numpy())
    return accuracy_score(all_labels, all_preds), total_loss/len(loader) if criterion else 0, all_labels, all_preds

# ============================================================================
# 10-FOLD CROSS VALIDATION (No Early Stopping)
# ============================================================================
def cross_validate_no_early_stop(X, y, folds=5):
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=42)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    fold_metrics = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        print(f"\n--- Fold {fold+1}/{folds} ---")

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        train_loader = DataLoader(list(zip(X_train, y_train)), BATCH_SIZE, shuffle=True)
        test_loader = DataLoader(list(zip(X_test, y_test)), BATCH_SIZE)

        model = CTRGCN().to(device)
        criterion = nn.CrossEntropyLoss(label_smoothing=0.20)
        optimizer = Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=5e-2)

        # TRAINING LOOP
        for epoch in range(EPOCHS):
            model.train()
            for data, target in train_loader:
                data = SkeletonUtils.augment(data)
                data, target = data.to(device), target.to(device)
                optimizer.zero_grad()
                output = model(data)
                loss = criterion(output, target)
                loss.backward()
                optimizer.step()

            if (epoch+1) % 10 == 0 or epoch == 0:
                train_acc, _, _, _ = evaluate(model, train_loader, device)
                print(f"Epoch {epoch+1}/{EPOCHS} | Train Acc: {train_acc:.2%}")

        # Evaluate on test set
        test_acc, _, y_true, y_pred = evaluate(model, test_loader, device)
        print(f"Fold {fold+1} Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
        fold_metrics.append(test_acc)

    print(f"\nAverage Test Accuracy over {folds} folds: {np.mean(fold_metrics):.4f} ({np.mean(fold_metrics)*100:.2f}%)")
    print(f"Standard Deviation: {np.std(fold_metrics):.4f}")

# ============================================================================
# MAIN FUNCTION
# ============================================================================
def main_cv_no_early_stop():
    proc = SquatProcessor()
    videos = glob.glob(os.path.join(VIDEO_DIR, "*.mp4"))
    X, y = [], []

    print("Extracting features from videos...")
    for v in videos:
        coords, label, _ = proc.process_video(v)
        if coords is not None:
            for i in range(0, len(coords) - SEQ_LEN, 10):
                X.append(coords[i:i+SEQ_LEN].transpose(2, 0, 1))
                y.append(label)

    if not X:
        print("No valid data found. Check your VIDEO_DIR path.")
        return

    X = torch.FloatTensor(np.array(X))
    y = torch.LongTensor(np.array(y))

    cross_validate_no_early_stop(X, y, folds=5)

# ============================================================================
# RUN
# ============================================================================
if __name__ == "__main__":
    main_cv_no_early_stop()

Extracting features from videos...

--- Fold 1/5 ---
Epoch 1/100 | Train Acc: 72.41%
Epoch 10/100 | Train Acc: 72.41%
Epoch 20/100 | Train Acc: 77.59%
Epoch 30/100 | Train Acc: 81.90%
Epoch 40/100 | Train Acc: 86.21%
Epoch 50/100 | Train Acc: 90.52%
Epoch 60/100 | Train Acc: 92.24%
Epoch 70/100 | Train Acc: 90.52%
Epoch 80/100 | Train Acc: 86.21%
Epoch 90/100 | Train Acc: 100.00%
Epoch 100/100 | Train Acc: 75.86%
Fold 1 Test Accuracy: 0.7333 (73.33%)

--- Fold 2/5 ---
Epoch 1/100 | Train Acc: 71.79%
Epoch 10/100 | Train Acc: 72.65%
Epoch 20/100 | Train Acc: 84.62%
Epoch 30/100 | Train Acc: 99.15%
Epoch 40/100 | Train Acc: 95.73%
Epoch 50/100 | Train Acc: 100.00%
Epoch 60/100 | Train Acc: 100.00%
Epoch 70/100 | Train Acc: 100.00%
Epoch 80/100 | Train Acc: 100.00%
Epoch 90/100 | Train Acc: 100.00%
Epoch 100/100 | Train Acc: 100.00%
Fold 2 Test Accuracy: 1.0000 (100.00%)

--- Fold 3/5 ---
Epoch 1/100 | Train Acc: 71.79%
Epoch 10/100 | Train Acc: 80.34%
Epoch 20/100 | Train Acc: 91.45%
Epo